# 05 — DGIdb & Open Targets Cross-Check

**Stage 6:** cross-check our EGFR drugs against two independent sources.
- **DGIdb** — drug–gene interactions (do other databases agree our drugs hit EGFR?)
- **Open Targets** — which diseases EGFR is associated with (e.g. lung cancers)

Outputs: `egfr_dgidb_interactions.csv`, `egfr_opentargets_associations.csv`, `egfr_external_crosscheck_summary.csv`

### 1. Test notebook environment

In [ ]:
import sys
import time
import re
from pathlib import Path

import requests
import pandas as pd

print("Notebook is working")
print("Python executable:", sys.executable)

### 2. Set project folders

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)

### 3. Load existing dataset files

In [ ]:
recommendations_file = PROCESSED_DIR / "egfr_drug_recommendations.csv"
pubmed_summary_file = PROCESSED_DIR / "egfr_pubmed_summary.csv"
clinical_trials_summary_file = PROCESSED_DIR / "egfr_clinical_trials_summary.csv"
openfda_summary_file = PROCESSED_DIR / "egfr_openfda_summary.csv"

if not recommendations_file.exists():
    raise FileNotFoundError("egfr_drug_recommendations.csv not found. Run Notebook 01 first.")

drug_recommendations_df = pd.read_csv(recommendations_file)
print("Drug recommendations:", len(drug_recommendations_df))

for name, fp in {
    "PubMed summary": pubmed_summary_file,
    "Clinical trials summary": clinical_trials_summary_file,
    "openFDA summary": openfda_summary_file,
}.items():
    print(name, "exists:", fp.exists())

drug_recommendations_df.head()

### 4. Set target metadata

In [ ]:
target_name = "EGFR"
target_full_name = "Epidermal growth factor receptor"
target_chembl_id = "CHEMBL203"
target_ensembl_id = "ENSG00000146648"  # EGFR Ensembl gene id (Open Targets)

print("Target name:", target_name)
print("Target full name:", target_full_name)
print("Target ChEMBL ID:", target_chembl_id)
print("Target Ensembl ID:", target_ensembl_id)

### 5. Helper functions

In [ ]:
def normalize_name(value):
    """Normalise drug names for safer matching (uppercase, strip non-alphanumerics)."""
    if pd.isna(value):
        return ""
    value = str(value).upper()
    return re.sub(r"[^A-Z0-9]+", "", value)


def graphql_post(url, query, variables=None, retries=3, pause=1):
    """POST to a GraphQL endpoint with simple retries."""
    payload = {"query": query, "variables": variables or {}}
    for attempt in range(retries):
        try:
            response = requests.post(url, json=payload, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return None

### 6. Prepare project drug names

In [ ]:
if "drug_name" not in drug_recommendations_df.columns:
    raise ValueError("drug_name column not found in egfr_drug_recommendations.csv")

project_drugs_df = (
    drug_recommendations_df[["drug_name"]].dropna().drop_duplicates().copy()
)
project_drugs_df["normalised_drug_name"] = project_drugs_df["drug_name"].apply(normalize_name)
project_drug_names = set(project_drugs_df["normalised_drug_name"])

print("Number of project drugs:", len(project_drugs_df))
project_drugs_df.head(20)

### 7. Query DGIdb for EGFR drug-gene interactions
Note: in the current DGIdb schema, `interactions` is a direct list (no `nodes` wrapper).

In [ ]:
DGIDB_GRAPHQL_URL = "https://dgidb.org/api/graphql"

dgidb_query = """query interactionsByGene($geneNames: [String!]!) {
  genes(names: $geneNames) {
    nodes {
      name
      conceptId
      interactions {
        interactionScore
        drug { name conceptId }
        interactionTypes { type directionality }
        sources { sourceDbName }
      }
    }
  }
}"""

dgidb_response = graphql_post(
    url=DGIDB_GRAPHQL_URL,
    query=dgidb_query,
    variables={"geneNames": [target_name]},
)

if dgidb_response is None:
    print("DGIdb request failed.")
elif "errors" in dgidb_response:
    print("DGIdb returned GraphQL errors.")
    print(dgidb_response["errors"][:2])
else:
    print("DGIdb response received.")
    print(dgidb_response.keys())

### 8. Convert DGIdb response into a dataframe

In [ ]:
dgidb_records = []

try:
    gene_nodes = (dgidb_response or {}).get("data", {}).get("genes", {}).get("nodes", [])
    for gene in gene_nodes:
        gene_name = gene.get("name")
        gene_concept_id = gene.get("conceptId")
        for interaction in gene.get("interactions", []) or []:
            drug = interaction.get("drug", {}) or {}
            itypes = interaction.get("interactionTypes", []) or []
            sources = interaction.get("sources", []) or []
            dgidb_records.append({
                "target_name": target_name,
                "gene_name": gene_name,
                "gene_concept_id": gene_concept_id,
                "drug_name": drug.get("name"),
                "drug_concept_id": drug.get("conceptId"),
                "interaction_score": interaction.get("interactionScore"),
                "interaction_types": " | ".join(i.get("type", "") for i in itypes if i.get("type")),
                "sources": " | ".join(s.get("sourceDbName", "") for s in sources if s.get("sourceDbName")),
                "source": "DGIdb",
            })
except Exception as error:
    print("Could not parse DGIdb response:", error)

dgidb_interactions_df = pd.DataFrame(dgidb_records)
if dgidb_interactions_df.empty:
    dgidb_interactions_df = pd.DataFrame(columns=[
        "target_name", "gene_name", "gene_concept_id", "drug_name", "drug_concept_id",
        "interaction_score", "interaction_types", "sources", "source"])

print("DGIdb interaction records:", len(dgidb_interactions_df))
dgidb_interactions_df.head(20)

### 9. Match DGIdb drugs to our project drug list

In [ ]:
if not dgidb_interactions_df.empty:
    dgidb_interactions_df["normalised_drug_name"] = dgidb_interactions_df["drug_name"].apply(normalize_name)
    dgidb_interactions_df["matches_project_drug"] = dgidb_interactions_df["normalised_drug_name"].isin(project_drug_names)
else:
    dgidb_interactions_df["normalised_drug_name"] = pd.Series(dtype="object")
    dgidb_interactions_df["matches_project_drug"] = pd.Series(dtype="bool")

matched_dgidb_df = dgidb_interactions_df[dgidb_interactions_df["matches_project_drug"] == True].copy()
print("DGIdb records matching project drugs:", len(matched_dgidb_df))
matched_dgidb_df.head(20)

### 10. Save DGIdb interactions

In [ ]:
dgidb_interactions_file = PROCESSED_DIR / "egfr_dgidb_interactions.csv"
dgidb_interactions_df.to_csv(dgidb_interactions_file, index=False)
print("Saved:", dgidb_interactions_file)

### 11. Query Open Targets for EGFR disease associations

In [ ]:
OPENTARGETS_GRAPHQL_URL = "https://api.platform.opentargets.org/api/v4/graphql"

opentargets_query = """query targetAssociatedDiseases($ensemblId: String!, $size: Int!) {
  target(ensemblId: $ensemblId) {
    id
    approvedSymbol
    approvedName
    associatedDiseases(page: { index: 0, size: $size }) {
      count
      rows { score disease { id name } }
    }
  }
}"""

opentargets_response = graphql_post(
    url=OPENTARGETS_GRAPHQL_URL,
    query=opentargets_query,
    variables={"ensemblId": target_ensembl_id, "size": 20},
)

if opentargets_response is None:
    raise RuntimeError("Open Targets request failed.")
if "errors" in opentargets_response:
    raise RuntimeError(opentargets_response["errors"])

print("Open Targets response received.")
print(opentargets_response.keys())

### 12. Convert Open Targets response into a dataframe

In [ ]:
target_data = (opentargets_response or {}).get("data", {}).get("target", {}) or {}
associated_diseases = target_data.get("associatedDiseases", {}) or {}
association_rows = associated_diseases.get("rows", []) or []

opentargets_records = []
for row in association_rows:
    disease = row.get("disease", {}) or {}
    opentargets_records.append({
        "target_name": target_name,
        "target_full_name": target_data.get("approvedName"),
        "target_ensembl_id": target_data.get("id"),
        "approved_symbol": target_data.get("approvedSymbol"),
        "disease_id": disease.get("id"),
        "disease_name": disease.get("name"),
        "association_score": row.get("score"),
        "source": "Open Targets",
    })

opentargets_associations_df = pd.DataFrame(opentargets_records)
print("Total associated diseases (Open Targets):", associated_diseases.get("count"))
print("Rows fetched:", len(opentargets_associations_df))
opentargets_associations_df.head(20)

### 13. Save Open Targets associations

In [ ]:
opentargets_associations_file = PROCESSED_DIR / "egfr_opentargets_associations.csv"
opentargets_associations_df.to_csv(opentargets_associations_file, index=False)
print("Saved:", opentargets_associations_file)

### 14. Build the external cross-check summary

In [ ]:
dgidb_interaction_count = len(dgidb_interactions_df)
matched_project_drug_count = len(matched_dgidb_df)
opentargets_associated_disease_count = associated_diseases.get("count", len(opentargets_associations_df))

if not opentargets_associations_df.empty:
    top_opentargets_diseases = " | ".join(
        opentargets_associations_df.sort_values("association_score", ascending=False)["disease_name"].dropna().head(5).tolist()
    )
else:
    top_opentargets_diseases = ""

external_crosscheck_summary_df = pd.DataFrame([{
    "target_name": target_name,
    "target_full_name": target_full_name,
    "target_chembl_id": target_chembl_id,
    "target_ensembl_id": target_ensembl_id,
    "dgidb_interaction_count": dgidb_interaction_count,
    "matched_project_drug_count": matched_project_drug_count,
    "opentargets_associated_disease_count": opentargets_associated_disease_count,
    "top_opentargets_diseases": top_opentargets_diseases,
}])
external_crosscheck_summary_df

### 15. Add an external evidence score

In [ ]:
def calculate_external_crosscheck_score(row):
    score = 0.0
    if row["dgidb_interaction_count"] > 0:
        score += 0.3
    if row["matched_project_drug_count"] > 0:
        score += 0.3
    if row["opentargets_associated_disease_count"] > 0:
        score += 0.3
    if row["opentargets_associated_disease_count"] >= 10:
        score += 0.1
    return round(min(score, 1.0), 2)


external_crosscheck_summary_df["external_crosscheck_score"] = external_crosscheck_summary_df.apply(
    calculate_external_crosscheck_score, axis=1
)
external_crosscheck_summary_df

### 16. Save external cross-check summary

In [ ]:
external_summary_file = PROCESSED_DIR / "egfr_external_crosscheck_summary.csv"
external_crosscheck_summary_df.to_csv(external_summary_file, index=False)
print("Saved:", external_summary_file)

### 17. Final result

In [ ]:
print("DGIdb and Open Targets Cross-Check Complete")
print("=" * 70)
print("Target:", target_name)
print("DGIdb interaction records:", len(dgidb_interactions_df))
print("DGIdb records matching project drugs:", len(matched_dgidb_df))
print("Open Targets disease rows fetched:", len(opentargets_associations_df))
print("Open Targets total associated disease count:", associated_diseases.get("count"))
display(external_crosscheck_summary_df)
display(dgidb_interactions_df.head(10))
display(opentargets_associations_df.head(10))